### 02.- Ingest Reference Data
Ingesta de tablas de referencia (MCC Codes y Fraud Labels) desde la capa RAW hacia Bronze.


In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container",       "raw")
dbutils.widgets.text("containerDestino","bronze")
dbutils.widgets.text("catalogo",        "bank_dev")
dbutils.widgets.text("esquema",         "bronze")
dbutils.widgets.text("storageName",     "saprojectbankdeveastus")

In [0]:
container        = dbutils.widgets.get("container")
containerDestino = dbutils.widgets.get("containerDestino")
catalogo         = dbutils.widgets.get("catalogo")
esquema          = dbutils.widgets.get("esquema")
storageName      = dbutils.widgets.get("storageName")

In [0]:
storage_path         = f"abfss://{container}@{storageName}.dfs.core.windows.net/"
storage_path_destino = f"abfss://{containerDestino}@{storageName}.dfs.core.windows.net/"

# Fuentes (RAW) con comodines para automatizacion continua
raw_mcc_codes_root     = f"{storage_path}external/catalogos/*/mcc_codes.json"
raw_fraud_labels_root  = f"{storage_path}external/catalogos/*/train_fraud_labels.json"

# Checkpoints y Schema Locations (BRONZE)
schema_location_mcc_codes        = f"{storage_path_destino}_schemas/mcc_codes"
checkpoint_location_mcc_codes    = f"{storage_path_destino}_checkpoints/mcc_codes"

schema_location_fraud_labels     = f"{storage_path_destino}_schemas/fraud_labels"
checkpoint_location_fraud_labels = f"{storage_path_destino}_checkpoints/fraud_labels"



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.types import MapType, StringType, LongType

## MCC Codes

In [0]:
# 1. Leer el JSON como texto crudo (todo el archivo en una sola celda/fila)
df_mcc_raw = (
    spark.read
    .format("text")
    .option("wholetext", "true")
    .load(raw_mcc_codes_root)
)

# 2. Parsear el diccionario dinámico y transformarlo en las columnas de tu tabla
df_mcc_final = (
    df_mcc_raw
    # Convertimos el texto a un mapa de Spark (Clave: Valor)
    .withColumn("json_map", F.from_json(F.col("value"), "MAP<STRING, STRING>"))
    # Explotamos el mapa para generar las filas de tu esquema
    .select(F.explode("json_map").alias("mcc", "description"))
    # Agregamos la columna de auditoría requerida por tu DDL
    .withColumn("ingestion_date", F.current_timestamp())
)

# 3. Guardar en tu tabla Bronze (Sobreescribiendo de forma segura)
(
    df_mcc_final
    .select("mcc", "description", "ingestion_date") # Aseguramos el orden exacto
    # dropDuplicates no va en Bronze — el dict JSON garantiza claves únicas por diseño.
    # Si hubiera duplicados reales, se detectan y eliminan en Silver.
    .write
    .format("delta")
    .mode("overwrite")         # referencia estática: overwrite completo es correcto
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{esquema}.mcc_codes")
)

## Fraud Labels

In [0]:
# 1. Leer el JSON como texto crudo (todo el archivo en una sola celda)
df_fraud_raw = (
    spark.read
    .format("text")
    .option("wholetext", "true")
    .load(raw_fraud_labels_root)
)

# 2. Parsear el nodo "target", explotarlo y aplicar el tipado correcto
df_fraud_final = (
    df_fraud_raw
    # Parseamos la estructura buscando el mapa interno dentro de "target"
    .withColumn("json_struct", F.from_json(F.col("value"), "target MAP<STRING, STRING>"))
    # Explotamos para generar las columnas clave-valor temporales
    .select(F.explode("json_struct.target").alias("key_id", "value_target"))
    # Transformamos al esquema exacto de tu DDL
    .select(
        F.col("key_id").alias("transaction_id"),  # STRING en Bronze — cast a BIGINT en Silver
        F.col("value_target").alias("target"),                   # "Yes" / "No" como STRING
        F.current_timestamp().alias("ingestion_date")            # TIMESTAMP de auditoría
    )
)

# 3. Guardar de forma segura en la tabla Bronze usando tus widgets
(
    df_fraud_final
    .select("transaction_id", "target", "ingestion_date")
    # dropDuplicates no va en Bronze — deduplicación es responsabilidad de Silver.
    .write
    .format("delta")
    .mode("overwrite")         # referencia estática: overwrite completo es correcto
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{esquema}.fraud_labels")
)